# 02 Descriptor Quality Control

**Scope.** Check RDKit, QM, RDKit+QM, and ECFP4 descriptor resources for shape consistency, missing values, basic feature health, and configuration alignment without generating derived outputs by default.

## Expected input files

- `data/processed/benchmark_molecules_4495.csv`
- `data/processed/rdkit_descriptors.csv`
- `data/processed/qm_descriptors.csv`
- `data/processed/rdkit_qm_descriptors.csv`
- `data/processed/ecfp4_fingerprints.npz`
- `data/processed/feature_scaler_parameters.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt

SAVE_OUTPUTS = False


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_configs = (candidate / "configs").is_dir()
        has_readme = (candidate / "README.md").exists()
        if has_configs and has_readme:
            return candidate
        if has_configs:
            fallback = candidate
            return fallback
    raise FileNotFoundError("Could not locate project root containing configs/.")


def load_yaml(path: Path):
    with path.open("r", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def check_files(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({"path": path.as_posix(), "exists": path.exists()})
    return pd.DataFrame(rows)


def require_files(paths):
    missing = [Path(path) for path in paths if not Path(path).exists()]
    if missing:
        formatted = "
".join(f"- {path.as_posix()}" for path in missing)
        raise FileNotFoundError(
            "The following required files are missing:
"
            f"{formatted}
"
            "Data required from user later before this notebook can run end-to-end."
        )


def print_missing_file_report(report_df: pd.DataFrame) -> None:
    missing = report_df.loc[~report_df["exists"], "path"].tolist()
    if missing:
        print("Data required from user later:")
        for item in missing:
            print(f"- {item}")
    else:
        print("All expected input files are present.")


PROJECT_ROOT = find_project_root()
CONFIG_DIR = PROJECT_ROOT / "configs"
DATASET_CFG = load_yaml(CONFIG_DIR / "dataset.yaml")


## 1. Scope

This notebook performs descriptor-table quality-control checks for benchmark molecular representations. It should report missing files clearly and must not fabricate descriptor summaries if the underlying processed tables are absent.

## 2. Expected inputs

The notebook expects benchmark molecule metadata, descriptor tables, fingerprint storage, and saved scaling parameters.

## 3. Load configs

In [ ]:
descriptor_cfg = load_yaml(CONFIG_DIR / 'descriptor_sets.yaml')
qm_cfg = load_yaml(CONFIG_DIR / 'qm_workflow.yaml')
expected_files = [
    PROJECT_ROOT / DATASET_CFG['files']['benchmark_molecules'],
    PROJECT_ROOT / DATASET_CFG['files']['rdkit_descriptors'],
    PROJECT_ROOT / DATASET_CFG['files']['qm_descriptors'],
    PROJECT_ROOT / DATASET_CFG['files']['rdkit_qm_descriptors'],
    PROJECT_ROOT / DATASET_CFG['files']['ecfp4_fingerprints'],
    PROJECT_ROOT / DATASET_CFG['files']['feature_scaler_parameters'],
]
print('Loaded descriptor configuration keys:', list(descriptor_cfg['representations'].keys()))

## 4. File existence checks

In [ ]:
file_report = check_files(expected_files)
display(file_report)
print_missing_file_report(file_report)

## 5. Descriptor table shape checks

In [ ]:
csv_files = {
    'benchmark': expected_files[0],
    'rdkit': expected_files[1],
    'qm': expected_files[2],
    'rdkit_qm': expected_files[3],
    'scaler': expected_files[5],
}
loaded = {}
if all(path.exists() for path in csv_files.values()):
    for name, path in csv_files.items():
        loaded[name] = pd.read_csv(path)
    shape_summary = pd.DataFrame([
        {'table': name, 'n_rows': df.shape[0], 'n_cols': df.shape[1]}
        for name, df in loaded.items()
    ])
    display(shape_summary)
else:
    print('Descriptor shape checks require the CSV inputs listed above.')

if expected_files[4].exists():
    with np.load(expected_files[4], allow_pickle=True) as fp:
        print('ECFP4 fingerprint archive keys:', list(fp.files))
else:
    print('Fingerprint archive missing; NPZ-specific checks skipped.')

## 6. Molecule ID alignment checks

In [ ]:
if {'benchmark', 'rdkit', 'qm', 'rdkit_qm'} <= set(loaded):
    benchmark_ids = loaded['benchmark']['molecule_id'].astype(str)
    alignment = []
    for name in ['rdkit', 'qm', 'rdkit_qm']:
        has_id = 'molecule_id' in loaded[name].columns
        aligned = has_id and benchmark_ids.tolist() == loaded[name]['molecule_id'].astype(str).tolist()
        alignment.append({'table': name, 'has_molecule_id': has_id, 'aligned_to_benchmark': aligned})
    display(pd.DataFrame(alignment))
else:
    print('Molecule ID alignment checks are pending input availability.')

## 7. Missing value checks

In [ ]:
if {'rdkit', 'qm', 'rdkit_qm'} <= set(loaded):
    missing_summary = []
    for name in ['rdkit', 'qm', 'rdkit_qm']:
        feature_df = loaded[name].drop(columns=['molecule_id'], errors='ignore')
        missing_summary.append({
            'table': name,
            'n_columns_with_missing': int(feature_df.isna().any(axis=0).sum()),
            'n_rows_with_missing': int(feature_df.isna().any(axis=1).sum()),
        })
    display(pd.DataFrame(missing_summary))
else:
    print('Missing-value checks require descriptor tables.')

## 8. Constant-column checks

In [ ]:
if {'rdkit', 'qm', 'rdkit_qm'} <= set(loaded):
    constant_summary = []
    for name in ['rdkit', 'qm', 'rdkit_qm']:
        feature_df = loaded[name].drop(columns=['molecule_id'], errors='ignore')
        constant_cols = feature_df.columns[feature_df.nunique(dropna=False) <= 1].tolist()
        constant_summary.append({'table': name, 'n_constant_columns': len(constant_cols), 'example_columns': constant_cols[:10]})
    display(pd.DataFrame(constant_summary))
else:
    print('Constant-column checks require descriptor tables.')

## 9. Correlation screening preview for RDKit descriptors

In [ ]:
if 'rdkit' in loaded:
    rdkit_features = loaded['rdkit'].drop(columns=['molecule_id'], errors='ignore').select_dtypes(include=[np.number])
    if rdkit_features.shape[1] >= 2:
        corr = rdkit_features.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        high_corr_pairs = (
            upper.stack()
            .rename('abs_r')
            .reset_index()
            .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b'})
        )
        threshold = descriptor_cfg['representations']['rdkit']['correlation_filter']['threshold_abs_r']
        display(high_corr_pairs[high_corr_pairs['abs_r'] >= threshold].sort_values('abs_r', ascending=False).head(20))
    else:
        print('Not enough numeric RDKit features for a correlation preview.')
else:
    print('RDKit descriptor table required for correlation screening preview.')

## 10. QM descriptor category coverage check based on `descriptor_sets.yaml`

In [ ]:
if 'qm' in loaded:
    expected_qm = []
    for category, names in descriptor_cfg['representations']['qm']['categories'].items():
        for name in names:
            expected_qm.append({'category': category, 'descriptor': name, 'present': name in loaded['qm'].columns})
    coverage_df = pd.DataFrame(expected_qm)
    display(coverage_df)
    print('Missing expected QM descriptors:', int((~coverage_df['present']).sum()))
else:
    print('QM descriptor coverage check requires the QM descriptor table.')

## 11. Scaling-parameter sanity check

In [ ]:
if 'scaler' in loaded:
    scaler = loaded['scaler']
    expected_fields = {'feature_name', 'mean', 'std'}
    print('Scaler columns:', scaler.columns.tolist())
    print('Contains expected fields:', expected_fields.issubset(set(scaler.columns)))
else:
    print('Scaling-parameter sanity check requires feature_scaler_parameters.csv.')

## 12. Missing data checklist

In [ ]:
missing_items = file_report.loc[~file_report['exists'], 'path'].tolist()
if missing_items:
    print('Data required from user later:')
    for item in missing_items:
        print(f'- {item}')
else:
    print('No missing descriptor QC input files detected for this notebook.')